In [ ]:
import sys 

# Please do not remove package declarations because these are used by the autograder.
def profile_most_probable_kmer(text: str, k: int,
                               profile: list[dict[str, float]]) -> str:
    """Identifies the most probable k-mer according to a given profile matrix.

    The profile matrix is represented as a list of columns, where the i-th element is a map
    whose keys are strings ("A", "C", "G", "T") and whose values represent the probability
    associated with this symbol in the i-th column of the profile matrix.
    """
    best_str=""
    best_score=0
    # dict has A,C,T,G
    for i in range(len(text)-k+1):
        current_score=1
        current_str=text[i:i+k]
        for j in range(k):
            curr_nt=current_str[j]
            
            score = profile[j][curr_nt]
            current_score*=score
        if current_score>best_score:
            best_str=current_str
            best_score=current_score
                
    return best_str
                
def make_profile(motifs):
    profile=[]
    for motif in motifs:
        for i in range(len(motif)):
            while len(profile)<=i:
                profile.append({"A":1,"T":1,"C":1,"G":1})
            nt=motif[i]
            profile[i][nt]=profile[i].get(nt,1)+1

    # print(profile)


    for j in range(len(profile)):
        for k,v in profile[j].items():
            profile[j][k] = v/(2*len(motifs))
            # print(v)
    return profile

def get_score(motifs):
    profile=[]
    for motif in motifs:
        for i in range(len(motif)):
            while len(profile)<=i:
                profile.append({"A":0,"T":0,"C":0,"G":0})
            nt=motif[i]
            profile[i][nt]=profile[i].get(nt,0)+1

    most_freq_nts=[]
    for i in range(len(profile)):
        most_freq_nt=max(profile[i],key=profile[i].get)

        most_freq_nts.append(most_freq_nt)
    # print(most_freq_nts)
    # print(motifs)
    # print(len(motifs[0]))
    # print((motif[0]))
    score_list=[0]*len(motifs[0])
    for motif in motifs:
        for i in range(len(motif)):
            if motif[i]!=most_freq_nts[i]:
                # print(score_list)
                score_list[i]+=1

    return (sum(score_list))

# Insert your gibbs_sampler function here, along with any subroutines you need
import random
def gibbs_sampling(dna: list[str], k: int, t: int, n: int) -> list[str]:
    motifs=[]
    for i in range(t):
        pos=random.randrange(0,len(dna[0])-k+1,1)
        # print(pos)
        motifs.append(dna[i][pos:pos+k])
    # return starting_motifs
    best_motifs=motifs.copy()

    for j in range(n):
        i=random.randrange(t)
        #print(i)
        profile=make_profile(motifs[:i]+motifs[i+1:])
        motif_i=profile_most_probable_kmer(dna[i],k,profile)
        #print(motif_i)
        #print(motifs)
        motifs[i]=motif_i
        #print(motifs)
        if get_score(motifs)<get_score(best_motifs):
            best_motifs=motifs.copy()
            #print("newpr")

    return best_motifs

def gibbs_sampler(dna: list[str], k: int, t: int, n: int) -> list[str]:
    """Implements the GibbsSampling algorithm for motif finding 1000 times"""
    times=1000
    best_output = gibbs_sampling(dna,k,t,n)
    best_score = get_score(best_output)

    while times>0:
        new_output = gibbs_sampling(dna,k,t,n)
        current_score = get_score(new_output)
        if current_score<best_score:
            best_score=current_score
            best_output=new_output.copy()
            
        times-=1

    return best_output
        

In [ ]:
# GibbsSampler(Dna, k, t, N)
#  randomly select k-mers Motifs = (Motif1, …, Motift) in each string from Dna
#  BestMotifs ← Motifs
#  for j ← 1 to N
#      i ← Random(t)
#      Profile ← profile matrix constructed from all strings in Motifs except for Motifi
#      Motifi ← Profile-randomly generated k-mer in the i-th sequence
#      if Score(Motifs) < Score(BestMotifs)
#          BestMotifs ← Motifs
#  return BestMotifs

# Input: Integers k, t, and N, followed by a collection of strings Dna.
# Output: The strings BestMotifs resulting from running GibbsSampler(Dna, k, t, N) with 1000 random starts. Remember to use pseudocounts!


# Test Cases
# Sample Input:

# 8 5 100
# CGCCCCTCTCGGGGGTGTTCAGTAAACGGCCA GGGCGAGGTATGTGTAAGTGCCAAGGTGCCAG TAGTACCGAGACCGAAAGAAGTATACAGGCGT TAGATCAAGTTTCAGGTGCACGTCGGTGAACC AATCCACCAGCTCCACGTGCAATGTTGGCCTA
# Sample Output:

# AACGGCCA AAGTGCCA TAGTACCG AAGTTTCA ACGTGCAA
